# Train2: fine-tune FunctionGemma on Colab

1. Upload `training/Train2/colab.zip` (made by `bash training/pack_colab.sh`) to the **root of My Drive**.
2. Runtime > Change runtime type > **GPU** (L4 or A100 preferred, T4 works but is slower).
3. Runtime > **Run all**. Allow access to Drive when asked.

When it finishes, the result is `MyDrive/train2_result.zip`: the trained model, the evaluation predictions and the logs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/fc && mkdir -p /content/fc && unzip -q /content/drive/MyDrive/colab.zip -d /content/fc
%cd /content/fc

In [ ]:
# keep Colab's torch
!pip install -q transformers==5.22.2 trl==1.17.1 datasets==5.1.0 accelerate==1.15.0

In [ ]:
!nvidia-smi
import torch
# under 20 GB (a T4 has 15 GB and no bf16): recompute activations to fit
low_memory = torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3
print("low memory GPU, gradient checkpointing on" if low_memory else "gradient checkpointing off")

In [ ]:
!mkdir -p training/Train2/output
flag = "--gradient-checkpointing" if low_memory else "--no-gradient-checkpointing"
!python training/train.py {flag} 2>&1 | tee training/Train2/output/train.log

In [ ]:
!python training/evaluate.py --models base,train2 --device cuda 2>&1 | tee training/Train2/output/eval.log

In [ ]:
!rm -f /content/train2_result.zip
!zip -r -q /content/train2_result.zip training/Train2/output/functiongemma training/Train2/output/eval_*.jsonl training/Train2/output/train.log training/Train2/output/eval.log -x 'training/Train2/output/functiongemma/checkpoint-*'
!cp /content/train2_result.zip /content/drive/MyDrive/train2_result.zip
!ls -lh /content/drive/MyDrive/train2_result.zip